# Phase 3 Execution Notebook

In [1]:
import pandas as pd
import numpy as np
import sklearn 
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
from data_handling import MrozHandler
import statsmodels.api as sm

norm = stats.norm

In [2]:
Mroz = MrozHandler("data/Mroz.csv")

Full dataset shape
Number of rows: 753
Number of features: 22
Working subset shape:
Number of rows: 428
Number of features: 22


#### Step 3.1: The Baseline Model (OLS Implementation)

##### step 3.1a - attach IMR vector as discussed in proposal 

In [3]:
# calculate IMR (Probability vector of choosing to work) using Probit:

Mroz.set_dependent("work", full=True)
Mroz.add_independents("kidslt6", "nwifeinc", "educ", "agew", "motheduc", "fatheduc", full=True)
probit_result = sm.Probit(Mroz.get_y().astype(int), Mroz.get_X()).fit()
fitted = probit_result.fittedvalues
imr_values = norm.pdf(fitted) / norm.cdf(fitted)
imr_series = pd.Series(imr_values, index=Mroz.full.index)


Mroz.attach("IMR", imr_series, to_working=True) # attach result
Mroz.clear_caches()

Dependent variable set to: work
Independent variables: ['kidslt6', 'nwifeinc', 'educ', 'agew', 'motheduc', 'fatheduc']
Optimization terminated successfully.
         Current function value: 0.603822
         Iterations 5
Attached 'IMR' to dataset
Caches cleared


In [4]:
def heckman_object(Mroz: MrozHandler) -> tuple[pd.Series, pd.DataFrame, MrozHandler]:
    Mroz.clear_caches()
    Mroz.set_dependent("lwage", full=False)
    Mroz.add_independents("exper", "expersq", full=False)
    Mroz.add_controls("educ", "kidslt6", "nwifeinc", "IMR", full=False)
    y = Mroz.get_y().dropna() # Fix the log wage issue by dropping nans, which correspond to non-working individuals
    X = Mroz.get_X().loc[y.index] # Ensure X and y are aligned after dropping nans
    return y, X, Mroz

y, X, Mroz = heckman_object(Mroz)

print(f"Dependent variable (y): {y.name}, shape: {y.shape}")
print(f"Independent variables (X): {X.columns.tolist()}, shape: {X.shape}")

Caches cleared
Dependent variable set to: lwage
Independent variables: ['exper', 'expersq']
Control variables: ['educ', 'kidslt6', 'nwifeinc', 'IMR']
Dependent variable (y): lwage, shape: (428,)
Independent variables (X): ['const', 'exper', 'expersq', 'educ', 'kidslt6', 'nwifeinc', 'IMR'], shape: (428, 7)


In [6]:
# Manual Implementation of OLS with Heteroskedasticity-Robust Errors
import statsmodels.formula.api as smf

# Define the formula string: Y ~ D + X1 + X2
# Note: We use np.log() directly in the formula for Log-Level specification
formula_1 = 'lwage ~ exper + expersq + educ + kidslt6 + nwifeinc + IMR'

# Fit the model
model_1 = smf.ols(formula=formula_1, data=Mroz.full).fit(cov_type='HC1')

# Print the "Regression Anatomy"
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.163
Model:                            OLS   Adj. R-squared:                  0.151
Method:                 Least Squares   F-statistic:                     14.55
Date:                Mon, 09 Mar 2026   Prob (F-statistic):           4.15e-15
Time:                        17:30:26   Log-Likelihood:                -429.95
No. Observations:                 428   AIC:                             873.9
Df Residuals:                     421   BIC:                             902.3
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4342      0.413     -1.053      0.2

#### Step 3.2: Testing Heterogeneity (Interaction Terms)